[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/38_grpo_loss.ipynb)

# 🔴 Hard: GRPO Loss

Implement the **Group Relative Policy Optimization (GRPO)** loss — a group-wise, baseline-subtracted REINFORCE objective commonly used in RLAIF (reinforcement learning from AI feedback).

Given a batch of log-probabilities, scalar rewards, and group ids (one group per prompt), define the within-group normalized advantages:

$$A_i = \frac{r_i - \bar r_{g(i)}}{\text{std}_{g(i)} + \epsilon}$$

where \(\bar r_{g(i)}\) and \(\text{std}_{g(i)}\) are the mean and standard deviation of rewards in the group of example \(i\).

The GRPO loss is then the negative advantage-weighted log-probability:

$$\mathcal{L}_{\text{GRPO}} = -\mathbb{E}_i \big[\,\text{stop\_grad}(A_i)\, \log \pi_\theta(y_i)\big].$$

### Signature
```python
from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    """GRPO loss over a batch.

    logps: (B,) policy log-probs for each sampled response
    rewards: (B,) scalar rewards for each response
    group_ids: (B,) integers, same id = same prompt/group
    returns: scalar loss (Tensor)
    """
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn.functional as F

In [27]:
# ✏️ YOUR IMPLEMENTATION HERE

from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    # pass  # compute normalized advantages per group and return -mean(adv.detach() * logps)
    num_groups = group_ids.max().item() + 1 
    # unique_ids = group_ids.unique() 直接相当于set得到id序列
    
    loss = 0
    for gid in range(num_groups):
        gid_idlst = (group_ids == gid)
        gid_rewards, gid_logps = rewards[gid_idlst], logps[gid_idlst]
        # mean std
        gid_r_mean, gid_r_std = gid_rewards.mean(), gid_rewards.std(unbiased=False) 
        gid_adv = (gid_rewards-gid_r_mean)/(gid_r_std+eps)
        print(gid_adv)
        print(gid_logps)
        gid_adv_logps = gid_adv.detach() * gid_logps
        loss += -gid_adv_logps.mean()

    return loss / num_groups

def grpo_loss_ref(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    """Group Relative Policy Optimization (GRPO) loss.

    logps: (B,) policy log-probs for each sampled response
    rewards: (B,) scalar rewards for each response
    group_ids: (B,) integers, same id = same prompt/group
    returns: scalar loss (Tensor)
    """
    # Compute per-group normalized advantages A_i
    unique_ids = group_ids.unique()
    advantages = torch.empty_like(rewards)
    for gid in unique_ids:
        mask = group_ids == gid
        r_g = rewards[mask]
        mean_g = r_g.mean()
        std_g = r_g.std(unbiased=False)
        advantages[mask] = (r_g - mean_g) / (std_g + eps)
        # print((r_g - mean_g) / (std_g + eps))

    # Stop gradient through advantages
    advantages_detached = advantages.detach()
    print(advantages_detached)
    print(logps)
    # GRPO objective: -E[A_i * logpi_i]
    return -(advantages_detached * logps).mean()

In [28]:
# 🧪 Debug
logps = torch.tensor([0.0, -0.5, -1.0, -1.5])
rewards = torch.tensor([1.0, 0.8, 0.2, 0.0])
group_ids = torch.tensor([0, 0, 1, 1])
print('Loss:', grpo_loss(logps, rewards, group_ids).item())
print('Loss_ref:', grpo_loss_ref(logps, rewards, group_ids).item())

tensor([ 0.9999, -0.9999])
tensor([ 0.0000, -0.5000])
tensor([ 0.9999, -0.9999])
tensor([-1.0000, -1.5000])
Loss: -0.24997496604919434
tensor([ 0.9999, -0.9999,  0.9999, -0.9999])
tensor([ 0.0000, -0.5000, -1.0000, -1.5000])
Loss_ref: -0.24997496604919434


In [29]:
# ✅ SUBMIT
from torch_judge import check
check('grpo_loss')


🧪 Testing: GRPO (Group Relative Policy Optimization) Loss (Hard)
──────────────────────────────────────────────────
tensor([-1.3989,  0.5197,  0.8792])
tensor([ 0.9950, -0.7206, -0.1913], grad_fn=<IndexBackward0>)
tensor([ 1.1560, -1.2834,  0.1274])
tensor([-0.2141, -0.1857, -0.8173], grad_fn=<IndexBackward0>)
  ✅ [1/4] Basic shape & type (1.9ms)
tensor([ 0.9999, -0.9999])
tensor([ 0.0000, -0.5000])
tensor([ 0.9999, -0.9999])
tensor([-1.0000, -1.5000])
  ✅ [2/4] Numeric check vs reference (1.3ms)
tensor([-1.0000,  1.0000], grad_fn=<DivBackward0>)
tensor([-1.2892, -0.0070], grad_fn=<IndexBackward0>)
tensor([-1.0000,  1.0000], grad_fn=<DivBackward0>)
tensor([ 0.3155, -0.5631], grad_fn=<IndexBackward0>)
  ✅ [3/4] Gradient flows to logps only (1.1ms)
tensor([-1.0000,  1.0000])
tensor([0., 0.], grad_fn=<IndexBackward0>)
tensor([-1.0000,  1.0000])
tensor([0., 0.], grad_fn=<IndexBackward0>)
  ✅ [4/4] Group-wise normalization (1.0ms)
──────────────────────────────────────────────────
  🎉 All 